# 01 — Exploratory Data Analysis
M5 Walmart Sales Dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.data.preprocessor import load_raw_data, preprocess_pipeline, load_config

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

config = load_config('../config/config.yaml')

## 1. Load Raw Data

In [ ]:
sales, calendar, prices = load_raw_data(config)

print('=== Sales ===')
print(f'Shape: {sales.shape}')
print(f'Items: {sales["item_id"].nunique()}')
print(f'Stores: {sales["store_id"].nunique()}')
print(f'Categories: {sales["cat_id"].nunique()}')
print(f'Departments: {sales["dept_id"].nunique()}')
print(f'States: {sales["state_id"].nunique()}')
sales.head()

In [ ]:
print('=== Calendar ===')
print(f'Shape: {calendar.shape}')
print(f'Date range: {calendar["date"].min()} to {calendar["date"].max()}')
print(f'Events: {calendar["event_name_1"].dropna().nunique()} unique')
calendar.head()

In [ ]:
print('=== Sell Prices ===')
print(f'Shape: {prices.shape}')
print(f'Price range: ${prices["sell_price"].min():.2f} - ${prices["sell_price"].max():.2f}')
prices.head()

## 2. Sales Distribution

In [ ]:
d_cols = [c for c in sales.columns if c.startswith('d_')]
total_daily_sales = sales[d_cols].sum(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(total_daily_sales.values, linewidth=0.8)
axes[0].set_title('Total Daily Sales Across All Stores & Items')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Total Units Sold')

sales_flat = sales[d_cols].values.flatten()
sales_flat = sales_flat[sales_flat > 0]
axes[1].hist(sales_flat, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Non-Zero Daily Sales (per item-store)')
axes[1].set_xlabel('Units Sold')
axes[1].set_ylabel('Frequency')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 3. Sales by Category & Store

In [ ]:
cat_sales = sales.groupby('cat_id')[d_cols].sum().T
cat_sales.index = range(len(cat_sales))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for cat in cat_sales.columns:
    axes[0].plot(cat_sales[cat].rolling(28).mean(), label=cat, linewidth=1.2)
axes[0].set_title('28-Day Rolling Mean Sales by Category')
axes[0].legend()

store_sales = sales.groupby('store_id')[d_cols].sum().T
store_totals = store_sales.sum().sort_values(ascending=True)
store_totals.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Total Sales by Store')
axes[1].set_xlabel('Total Units')

plt.tight_layout()
plt.show()

## 4. Seasonality Analysis

In [ ]:
cal_dates = pd.to_datetime(calendar['date'])
daily_total = pd.DataFrame({
    'date': cal_dates[:len(d_cols)],
    'total_sales': total_daily_sales.values
})
daily_total['dayofweek'] = daily_total['date'].dt.day_name()
daily_total['month'] = daily_total['date'].dt.month_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_sales = daily_total.groupby('dayofweek')['total_sales'].mean().reindex(dow_order)
dow_sales.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Average Sales by Day of Week')
axes[0].tick_params(axis='x', rotation=45)

month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_sales = daily_total.groupby('month')['total_sales'].mean().reindex(month_order)
month_sales.plot(kind='bar', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Average Sales by Month')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Zero Sales & Missing Data

In [ ]:
zero_pct = (sales[d_cols] == 0).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(zero_pct, bins=50, edgecolor='black', alpha=0.7, color='salmon')
axes[0].set_title('Distribution of Zero-Sales Percentage per Item-Store')
axes[0].set_xlabel('Fraction of Days with Zero Sales')
axes[0].axvline(zero_pct.median(), color='red', linestyle='--', label=f'Median: {zero_pct.median():.2f}')
axes[0].legend()

zero_by_cat = sales.groupby('cat_id').apply(
    lambda g: (g[d_cols] == 0).values.mean()
)
zero_by_cat.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_title('Zero-Sales Rate by Category')
axes[1].set_ylabel('Fraction of Zeros')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'\nOverall zero rate: {(sales[d_cols] == 0).values.mean():.3f}')
print(f'Missing prices: {prices["sell_price"].isna().sum()} / {len(prices)}')

## 6. Event & SNAP Impact

In [ ]:
cal_with_sales = calendar.iloc[:len(d_cols)].copy()
cal_with_sales['total_sales'] = total_daily_sales.values
cal_with_sales['has_event'] = cal_with_sales['event_name_1'].notna().astype(int)

event_impact = cal_with_sales.groupby('has_event')['total_sales'].mean()
print('Average daily sales:')
print(f'  No event:   {event_impact[0]:,.0f}')
print(f'  With event: {event_impact[1]:,.0f}')
print(f'  Lift:       {(event_impact[1]/event_impact[0] - 1)*100:.1f}%')

event_type_impact = cal_with_sales[cal_with_sales['event_name_1'].notna()].groupby(
    'event_type_1'
)['total_sales'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
event_type_impact.plot(kind='barh', ax=ax, color='goldenrod', edgecolor='black')
ax.axvline(event_impact[0], color='red', linestyle='--', label='No-event average')
ax.set_title('Average Sales by Event Type')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Price Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(prices['sell_price'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Distribution of Sell Prices')
axes[0].set_xlabel('Price ($)')
axes[0].set_yscale('log')

price_var = prices.groupby('item_id')['sell_price'].agg(['std', 'mean'])
price_var['cv'] = price_var['std'] / price_var['mean']
axes[1].hist(price_var['cv'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].set_title('Price Coefficient of Variation per Item')
axes[1].set_xlabel('CV (std / mean)')

plt.tight_layout()
plt.show()

print(f'Items with price changes: {(price_var["cv"] > 0.01).sum()} / {len(price_var)}')

## Summary
Key findings to carry into feature engineering and modeling:
- High zero-inflation — models need to handle intermittent demand
- Strong weekly seasonality (weekend dips)
- Events create measurable sales lift
- SNAP days affect state-level demand
- Price changes are infrequent but impactful